In [ ]:
import os
import glob
from typing import Dict, List, Tuple, Optional

import numpy as np
import xarray as xr

from pprint import pprint

import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from pcmdi_enso_reader import (
    ENSODiagReader
)

In [ ]:
class ENSODiversityPlotter:
    """
    Plot diagnostics for ENSO diversity metrics from `data_dict`.

    Expected data_dict structure:
        data_dict[var_name] = {
            "reference": DataArray(sample, ...),
            "hist":      DataArray(sample, ...),
            "future":    DataArray(sample, ...),
        }
    """

    def __init__(
        self,
        data_dict,
        target_group="ENSO_perf",
        target_metric="enso_sst_diversity_mode2",
        target_obs="ERA-Interim",
        fig_dir=".",
        fig_fmt="png",
    ):
        self.data_dict = data_dict
        self.target_group = target_group
        self.target_metric = target_metric
        self.target_obs = target_obs
        self.fig_dir = fig_dir
        self.fig_fmt = fig_fmt

        os.makedirs(self.fig_dir, exist_ok=True)

    # -----------------------------
    # Internal helpers
    # -----------------------------
    def _get_values(self, var, group):
        """Return 1D np.ndarray of non-NaN values for a given variable & group."""
        da = self.data_dict[var][group]
        vals = da.values.ravel()
        return vals[~np.isnan(vals)]

    def compute_custom_box_stats(self, vals, label):
        """
        Custom boxplot statistics for matplotlib.bxp:
          - q1  = 5th percentile
          - med = median
          - q3  = 95th percentile
          - whislo  = min(vals)
          - whishi  = max(vals)
        """
        vals = np.asarray(vals)
        vals = vals[~np.isnan(vals)]

        if vals.size == 0:
            raise ValueError(f"No valid data for {label} (all NaN or empty).")

        stats = {
            "label": label,
            "mean": float(np.mean(vals)),
            "med": float(np.median(vals)),
            "q1": float(np.percentile(vals, 5)),
            "q3": float(np.percentile(vals, 95)),
            "whislo": float(vals.min()),
            "whishi": float(vals.max()),
            "fliers": [],
        }
        return stats
        
    @staticmethod
    def _format_lon(lon):
        """
        Format longitude in 0–360 degrees as E/W.
        """
        lon = float(lon)
    
        if np.isnan(lon):
            return ""
    
        if lon == 0 or lon == 360:
            return "0°"
        if lon == 180:
            return "180°"
    
        if lon < 180:
            return f"{int(lon)}°E"
        else:
            return f"{int(360 - lon)}°W"

    # -----------------------------
    # Single-panel, all variables
    # -----------------------------
    def plot_enso_lon_box(
        self,
        variables=None,
        title=None,
        xlabel="",
        ylabel="Diagnostic value",
        ylim=None,           # (ymin, ymax) or None
        figsize=None,        # e.g., (10, 8)
        figgrps=None,  
        figdict=None,   
        show=False,
        save=True,
        fname=None,
        var_labels=None,     # dict: {var_name: label}
        font_size=12,
        # Groups and colors
    ):
        """
        Plot all variables in one panel using custom percentiles:
          - Q1 = 5th percentile
          - Q2 = median
          - Q3 = 95th percentile
          - whiskers = min, max
        """
        
        # Variables to plot
        if variables is None:
            variables = list(self.data_dict.keys())

        data_dict = self.data_dict
        obs_label = self.target_obs

        # Default title
        if title is None:
            title = (
                "ENSO Diversity Diagnostics\n"
                f"{self.target_group} / {self.target_metric} / Obs: {obs_label}"
            )
            
        ngroups = len(variables)
        width = 0.25
        x0 = np.arange(ngroups)

        if figsize is None:
            figsize = (1.6 * ngroups + 3, 6)

        fig, ax = plt.subplots(figsize=figsize)
        
        # Loop through variables and groups
        legend_labels = {}
        for i, var in enumerate(variables):
            for j, group in enumerate(figgrps):

                vals = self._get_values(var, group)
                stats = self.compute_custom_box_stats(vals, label=f"{var}-{group}")

                xpos = x0[i] + (j - 1) * width  # left/center/right offset
                color = figdict[group]['color']
                alpha = figdict[group]['alpha'] 
                if group not in legend_labels:
                    legend_labels[group] = figdict[group]['label']
                    
                bp = ax.bxp(
                    [stats],
                    positions=[xpos],
                    widths=width * 0.8,
                    showmeans=True,
                    meanprops=dict(
                        marker="o",
                        markerfacecolor=color,
                        markeredgecolor=color,
                        markersize=7,
                        markeredgewidth=0
                    ),
                    medianprops=dict(
                        color=color, 
                        linewidth=2
                    ),                 
                    patch_artist=True,
                )

                for patch in bp["boxes"]:
                    patch.set_facecolor(color)
                    patch.set_alpha(alpha)

        # x-axis labels (optionally mapped through var_labels)
        xticklabels = []
        for var in variables:
            if isinstance(var_labels, dict):
                xticklabels.append(var_labels.get(var, var))
            else:
                xticklabels.append(var)

        ax.set_xticks(x0)
        ax.set_xticklabels(xticklabels, rotation=0, ha="center", fontsize=font_size)

        # Axes labels and title
        ax.set_xlabel(xlabel, fontsize=font_size*1.0)
        ax.set_ylabel(ylabel, fontsize=font_size*1.0)
        ax.set_title(title, fontsize=font_size*1.1, loc = "left")
        ax.tick_params(axis="both", labelsize=fontz * 0.95)

        # y-limits
        if ylim is not None:
            ax.set_ylim(*ylim)
            
        yticks = ax.get_yticks()
        ax.set_yticks(yticks)   # ← REQUIRED
        ax.set_yticklabels([self._format_lon(t) for t in yticks], fontsize=font_size*0.95)

        # Legend
        handles = [
            Patch(facecolor=figdict[g]['color'], alpha=figdict[g]['alpha'] , edgecolor="black", label=g)
            for g in figgrps
        ]
        
        handle_labels = [ legend_labels[key] for key in legend_labels.keys() ] 
        ax.legend(handles, handle_labels, loc="best", fontsize=font_size*0.9)

        # Grid & layout
        ax.grid(True, linestyle="--", alpha=0.4)
        ax.tick_params(axis="both", labelsize=font_size - 2)
        fig.tight_layout()

        # Save
        if save:
            if fname is None:
                fname = f"enso_diversity_custom_boxplot.{self.fig_fmt}"
            outpath = os.path.join(self.fig_dir, fname)
            fig.savefig(outpath, dpi=150, bbox_inches="tight")
            print("Saved figure:", outpath)

        if show:
            plt.show()

        plt.close(fig)


In [ ]:
if __name__ == "__main__":
    TOP_DIR   = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper"
    DATA_DIR  = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le"
    OUT_DIR   = f"{TOP_DIR}/figure_data"
    FIG_DIR   = "./"
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(FIG_DIR, exist_ok=True)

    # ---- Pair experiments consistently (order matters) ----
    MODEL   = "v3.LR.historical"
    GROUP   = ["hist", "future"]
    PERIOD  = [(1985, 2014), (2015, 2049)]
    NENS    = [25, 25]

    members = None
    verbose = False
    diag_print = True 
    
    reader = ENSODiagReader(
        data_dir=DATA_DIR,
        model=MODEL,
        groups=GROUP,
        period_list=PERIOD,
        nens=NENS,
        members=members,
        verbose=verbose,
    )

    target_obs    = "ERA-Interim"
    target_group  = "ENSO_perf"
    target_metric = "enso_sst_diversity_mode2"
    target_var_list = [
        "Enso_lon_pos_maxSSTA",
        "Nina_lon_pos_minSSTA",
        "Nino_lon_pos_maxSSTA"
    ]

    # ---- Sanity checks ----
    print("Available ENSO groups:", reader.available_groups())
    print(f"Variables in {target_group}:", reader.available_vars(target_group))

    if target_group not in reader.available_groups():
        raise ValueError(f"Unknown target_group: {target_group}")

    if target_metric not in reader.available_vars(target_group):
        raise ValueError(f"Unknown target_metric '{target_metric}' for '{target_group}'")

    data_dict = {}   # model hist+ model future + obs 
    # ----------------------------------------------------
    # Loop through target variable list
    # ----------------------------------------------------
    for target_var in target_var_list:
        print(f"\n>>> Loading target_var = {target_var}")
        data_dict[target_var] = {} 
        
        # Load model + obs using your working function
        dm, do = reader.load_metric_data(
            enso_group=target_group,
            var_name=target_metric,
            nc_var=target_var,
            ref_tag=target_obs,
        )

        # observation only has one member and is identical for historical and near-future
        ref = reader.validate_constant_observation(
            do, ref_group="hist", ref_member="00",  sample_dim=None
        )
        # If we get here, all obs are identical
        # ref is the DataArray you want to keep
        data_dict[target_var]['reference'] = ref
        
        for group, member_dict in dm.items():
            print(f"  Group: {group}")
            pooled = reader.pool_members_to_samples(member_dict, sample_dim=None)  # sample_dim inferred
            data_dict[target_var][group] = pooled
            
    print("\n✓ All target_var successfully read.")

    if diag_print: 
        for var, group_dict in data_dict.items():
            print(f"\nVariable: {var}")
            for group, da in group_dict.items():
                print(f"  Group: {group}")
                # da is an xarray DataArray or Dataset
                shape = getattr(da, "shape", None)
                dims  = getattr(da, "dims", None)
                print(f"type={type(da).__name__} | shape={shape} | dims={dims}")

    # now do the plot
    figure_name = "enso_diversity_hist_future_ens.pdf"
    ymin=160
    ymax=300
    title = "(b) ENSO Diversity (SSTA Min/Max Center Longitude)"
    xlabel = ""
    ylabel = r"SSTA Min/Max Center Longitude ($^\circ$E/$^\circ$W)"
    fontz  = 18 
    figure_size = (10,8)
    fig_fmt="pdf"   # to match your figure_name extension
    show=True
    save=True

    plot_groups = ['reference'] + GROUP

    plot_dict = {        
        "reference":{
            "label": f"{target_obs} (Reference)",
            "color": "black",
            "alpha": 0.55,
        },
        "hist": {
            "label": "E3SM (Historical)",
            "color": "steelblue",
            "alpha": 0.55,
        },
        "future": {
            "label": "E3SM (Near-Future)",
            "color": "tomato",
            "alpha": 0.55,
        },
    }
    
    var_label_map = {
        "Enso_lon_pos_maxSSTA": "ENSO (max SSTA)",
        "Nina_lon_pos_minSSTA": "La Niña (min SSTA)",
        "Nino_lon_pos_maxSSTA": "El Niño max SSTA",
    }

    plotter = ENSODiversityPlotter(
        data_dict=data_dict,
        target_group=target_group,
        target_metric=target_metric,
        target_obs=target_obs,
        fig_dir=FIG_DIR,
        fig_fmt=fig_fmt, 
    )
    
    plotter.plot_enso_lon_box(
        variables=target_var_list,
        title=title,
        xlabel=xlabel,
        ylabel=ylabel,
        ylim=(ymin, ymax),
        figsize=figure_size,
        figdict=plot_dict,
        figgrps=plot_groups,
        show=show,
        save=save,
        fname=figure_name, 
        var_labels=var_label_map,
        font_size=fontz,
    )    
    